# Check for interference now that 2Dmesh params have been decided
 - smooth_tabin(50)
 - and smooth_taubin(50) plus remesh with fine_edge_length = 0.2 
    - but only for tpm-mc1

In [2]:
from phd_helpers.paths import (
    pose2idCMC, get_bone_transform, transform_mesh, get_task_stl_paths, get_mesh, get_info_df, get_trimesh, get_info
)

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
################# DATA #################
info = get_info_df()
cmc_info = info[info.group=='CMC']
stl_paths = get_task_stl_paths('CMC')
################# DATA #################

bones = np.array(['rad', 'uln', 'sca', 'lun', 'trq', 'pis', 'tpd',
                        'tpm', 'cap', 'ham', 'mc1', 'mc2', 'mc3', 'mc4', 'mc5'])
poses = ['adduction','abduction','flexion','extension','pinch','grasp',
                            'jar','pinch_load','grasp_load','jar_load','neutral']

smooth_iters = 50

In [3]:
bone_pairs = [ # articulating bone pairs
    ['tpm', 'mc1'],
    ['tpm', 'sca'],
    ['tpm', 'mc2'],
    ['tpm', 'tpd'],
    ['rad', 'uln'],
    ['rad', 'sca'],
    ['rad', 'lun'],
    ['sca', 'tpd'],
    ['sca', 'lun'],
    ['sca', 'cap'],
    ['lun', 'cap'],
    ['lun', 'ham'],
    ['lun', 'trq'],
    ['tpd', 'mc2'],
    ['tpd', 'cap'],
    ['cap', 'ham'],
    ['cap', 'mc2'],
    ['cap', 'mc3'],
    ['cap', 'mc4'],
    ['trq', 'ham'],
    ['trq', 'pis'],
    ['ham', 'mc4'],
    ['ham', 'mc5'],
    ['mc1', 'mc2'],
    ['mc2', 'mc3'],
    ['mc3', 'mc4'],
    ['mc4', 'mc5']
]

# Smooth

In [ ]:
################# STORE DATA #################
savedir = Path('Interference-smooth')
savedir.mkdir(parents=True, exist_ok=True)

intersections = {}
for stl_path in stl_paths:
    subject, sideL = get_info(stl_path)
    intersections[f'{subject}-{sideL}'] = {pose: [] for pose in poses}
################# STORE DATA #################

for j, stl_path in enumerate(stl_paths, start=1): 

    ################# SUBJECT INFO #################
    subject, sideL = get_info(stl_path)
    ################# SUBJECT INFO #################
    print(f"{subject}{sideL} {j}/{len(stl_paths)}")

    savepath = savedir / f'{subject+sideL}.csv' 
    if not savepath.is_file(): 

        ################# NEUTRAL BONE MESHES #################
        meshes_neu = []
        for bone in bones:
            meshes_neu.append(get_mesh(stl_path, bone).smooth_taubin(n_iter=smooth_iters))
        ################# NEUTRAL BONE MESHES #################


        for i, pose in enumerate(tqdm(poses)): 
                
                pose_id = pose2idCMC(pose)

                ################# TRANSFORM BONES #################
                meshes_posed = []
                for mesh, bone in zip(meshes_neu, bones):
                    try: # if subject has alternate neutral that will be used otherwise Exception and use default neutral
                        R, t = get_bone_transform(stl_path, bone, pose_id)
                    except:
                        R, t = np.eye(3), np.zeros(3)

                    meshes_posed.append(transform_mesh(mesh, R, t))
                ################# TRANSFORM BONES #################

                ################# CHECK INTERFERENCE #################
                for bone_pair in bone_pairs:
                    mesh1 = meshes_posed[np.where(bones==bone_pair[0])[0][0]]
                    mesh2 = meshes_posed[np.where(bones==bone_pair[1])[0][0]]

                    #intersection1 = np.sum(mesh1.select_enclosed_points(mesh2, tolerance=0)['SelectedPoints'])
                    #intersection2 = np.sum(mesh2.select_enclosed_points(mesh1, tolerance=0)['SelectedPoints'])
                    #intersection = np.sum(intersection1+intersection2) # doesn't catch if two triangle edges overlap
                    #intersections[f'{subject}-{sideL}'][poses[i]].append(intersection > 0)

                    # trimesh
                    trimesh1 = get_trimesh(mesh1)
                    trimesh2 = get_trimesh(mesh2)

                    intersection = trimesh1.intersection(trimesh2) # Check intersection
                    intersections[f'{subject}-{sideL}'][poses[i]].append(intersection.volume > 0) # poses in correct order!
                ################# CHECK INTERFERENCE #################

        ################# SAVE DATA (for each subject) #################
        subject_df = pd.DataFrame(intersections[f'{subject}-{sideL}'])
        subject_df['bone_pairs'] = bone_pairs
        subject_df['bone_pairs'] = subject_df['bone_pairs'].apply(lambda x: f'{x[0]}-{x[1]}')
        
        subject_df[subject_df.columns[-1:].to_list() + subject_df.columns[:-1].to_list()].to_csv(savepath, index=False)
        ################# SAVE DATA #################

In [6]:
from phd_helpers.paths import bone_pair_pose_intersection, get_project_root

In [22]:
poses = ['adduction','abduction','flexion','extension','pinch','grasp',
                            'jar','neutral']
pairs=[
    'tpm-mc1'
]

smooth = bone_pair_pose_intersection(pairs, poses, 'Interference-smooth')
original = bone_pair_pose_intersection(pairs, poses, get_project_root() / '../KineticsData/FEA/CheckInterference2')

print('\noriginal')
print(original.values)
print('\nsmooth')
print(smooth.values)


original
<StringArray>
['14873R', '50017L', '22306R', '50037L', '14613R', '50019R', '50034R',
 '14827L', '50016L', '50014R', '50027L', '14818R', '50029R', '50007L',
 '14726R', '50045R', '50053R', '50024R', '50049R', '50006R', '14548R',
 '15441R', '50008L', '50001R', '50021R', '14685R', '15283R', '50018L',
 '50020R', '50000R']
Length: 30, dtype: str

smooth
<StringArray>
['14873R', '50017L', '22306R', '50037L', '14613R', '50019R', '50034R',
 '14827L', '50016L', '50014R', '15737R', '50027L', '14727R', '14818R',
 '50029R', '50007L', '14819R', '14726R', '50045R', '14874R', '50053R',
 '50024R', '50049R', '15006R', '50006R', '14548R', '15441R', '50008L',
 '50001R', '50021R', '14685R', '15283R', '15294R', '50018L', '50020R',
 '50000R']
Length: 36, dtype: str


In [36]:
testing3 = np.array(['14818R', '14874R', '22306R', '50000R', '50049R', '50034R',
'15006R', '50017L', '15441R', '14827L', '14726R', '15294R', 
'50021R', '50027L', '50008L', '14548R', '14873R', '15737R', 
'14685R', '50045R', '50006R', '50029R', '50020R', '50018L', 
'50037L', '14727R', '50014R', '50019R', '50053R', '50016L', 
'50024R', '50007L', '50001R', '14819R', '15283R'])

print('missing from testing3 - (gave interference error in cartilage step)')
print(list(smooth.values[~np.isin(smooth.values, testing3)]))

missing from testing3 - (gave interference error in cartilage step)
['14613R']


In [26]:
print(len(testing3))

35


# Smooth and remesh

In [31]:
import subprocess
import pyvista as pv

In [32]:

path_MeshPipeline_main = '../../../../MeshPipeline/main.py'
subprocess.run(["python", path_MeshPipeline_main])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/MeshPipeline/set_parameters/parameters.json



SUBJECT: 50037L
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.852s - ok

SUBJECT: 50090R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 5.700s - ok

SUBJECT: 15294R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 4.261s - ok

SUBJECT: 50053R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 7.606s - ok

SUBJECT: 50049R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 5.620s - ok

SUBJECT: 15737R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 4.799s - ok

SUBJECT: 50061R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 7.870s - ok

SUBJECT: 14726R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 5.034s - ok

SUBJECT: 50016L
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.835s - ok

SUBJECT: 14613R
	BONES:

CompletedProcess(args=['python', '../../../../MeshPipeline/main.py'], returncode=0)

In [34]:

output_root = 'outputs/SmoothRemesh'
bone_arbone = 'tpm-mc1'
subject_sideL = '14548R'
run_id = '-0'

bone_remesh = pv.read(f'{output_root}/meshes/{subject_sideL}/{bone_arbone}/2Dmesh/bone_remesh{run_id}.obj')
bone_remesh.plot()

Widget(value='<iframe src="http://localhost:49403/index.html?ui=P_0x30dcbdf70_0&reconnect=auto" class="pyvista…

In [43]:
from phd_helpers.paths import (
    pose2idCMC, get_bone_transform, transform_mesh, get_task_stl_paths, get_mesh, get_info_df, get_trimesh, get_info
)

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
################# DATA #################
info = get_info_df()
cmc_info = info[info.group=='CMC']
stl_paths = get_task_stl_paths('CMC')
################# DATA #################

poses = ['adduction','abduction','flexion','extension','pinch','grasp',
                            'jar','pinch_load','grasp_load','jar_load','neutral']

smooth_iters = 50

In [ ]:
bone_pairs = np.array([
    ['tpm', 'mc1']
])
bones = bone_pairs[0]

################# STORE DATA #################
savedir = Path('Interference-SmoothRemesh')
savedir.mkdir(parents=True, exist_ok=True)

intersections = {}
for stl_path in stl_paths:
    subject, sideL = get_info(stl_path)
    intersections[f'{subject}-{sideL}'] = {pose: [] for pose in poses}
################# STORE DATA #################

for j, stl_path in enumerate(stl_paths, start=1): 

    ################# SUBJECT INFO #################
    subject, sideL = get_info(stl_path)
    subject_sideL = subject + sideL
    ################# SUBJECT INFO #################
    print(f"{subject}{sideL} {j}/{len(stl_paths)}")

    savepath = savedir / f'{subject+sideL}.csv' 
    if not savepath.is_file(): 

        ################# NEUTRAL BONE MESHES #################
        meshes_neu = [
            pv.read(f'{output_root}/meshes/{subject_sideL}/{bones[0]}-{bones[1]}/2Dmesh/bone_remesh{run_id}.obj'),
            pv.read(f'{output_root}/meshes/{subject_sideL}/{bones[1]}-{bones[0]}/2Dmesh/bone_remesh{run_id}.obj')
        ]
        ################# NEUTRAL BONE MESHES #################


        for i, pose in enumerate(tqdm(poses)): 
                
                pose_id = pose2idCMC(pose)

                ################# TRANSFORM BONES #################
                meshes_posed = []
                for mesh, bone in zip(meshes_neu, bones):
                    try: # if subject has alternate neutral that will be used otherwise Exception and use default neutral
                        R, t = get_bone_transform(stl_path, bone, pose_id)
                    except:
                        R, t = np.eye(3), np.zeros(3)

                    meshes_posed.append(transform_mesh(mesh, R, t))
                ################# TRANSFORM BONES #################

                ################# CHECK INTERFERENCE #################
                #for bone_pair in bone_pairs:
                mesh1 = meshes_posed[np.where(bones==bone_pair[0])[0][0]]
                mesh2 = meshes_posed[np.where(bones==bone_pair[1])[0][0]]

                #intersection1 = np.sum(mesh1.select_enclosed_points(mesh2, tolerance=0)['SelectedPoints'])
                #intersection2 = np.sum(mesh2.select_enclosed_points(mesh1, tolerance=0)['SelectedPoints'])
                #intersection = np.sum(intersection1+intersection2) # doesn't catch if two triangle edges overlap
                #intersections[f'{subject}-{sideL}'][poses[i]].append(intersection > 0)

                # trimesh
                trimesh1 = get_trimesh(mesh1)
                trimesh2 = get_trimesh(mesh2)

                intersection = trimesh1.intersection(trimesh2) # Check intersection
                intersections[f'{subject}-{sideL}'][poses[i]] = intersection.volume > 0 # poses in correct order!
                ################# CHECK INTERFERENCE #################

        ################# SAVE DATA (for each subject) #################
        #subject_df = pd.DataFrame(intersections[f'{subject}-{sideL}'])
        #subject_df['bone_pairs'] = bone_pair
        #subject_df['bone_pairs'] = subject_df['bone_pairs'].apply(lambda x: f'{x[0]}-{x[1]}')
        
        #subject_df[subject_df.columns[-1:].to_list() + subject_df.columns[:-1].to_list()].to_csv(savepath, index=False)
        ################# SAVE DATA #################
#pd.DataFrame(intersections).to_csv(savedir / 'tpm-mc1.csv')


50037L 1/46


  0%|          | 0/11 [00:00<?, ?it/s]/opt/miniconda3/envs/phd/lib/python3.12/site-packages/trimesh/triangles.py:302: RuntimeWarning: invalid value encountered in divide
  center_mass = integrated[1:4] / volume
100%|██████████| 11/11 [00:05<00:00,  2.03it/s]


50090R 2/46


  0%|          | 0/11 [00:00<?, ?it/s]/opt/miniconda3/envs/phd/lib/python3.12/site-packages/trimesh/triangles.py:302: RuntimeWarning: invalid value encountered in divide
  center_mass = integrated[1:4] / volume
100%|██████████| 11/11 [00:05<00:00,  2.03it/s]


15294R 3/46


...

  0%|          | 0/11 [00:00<?, ?it/s]/opt/miniconda3/envs/phd/lib/python3.12/site-packages/trimesh/triangles.py:302: RuntimeWarning: invalid value encountered in divide
  center_mass = integrated[1:4] / volume
100%|██████████| 11/11 [00:05<00:00,  1.92it/s]


50045R 45/46


  0%|          | 0/11 [00:00<?, ?it/s]/opt/miniconda3/envs/phd/lib/python3.12/site-packages/trimesh/triangles.py:302: RuntimeWarning: invalid value encountered in divide
  center_mass = integrated[1:4] / volume
100%|██████████| 11/11 [00:05<00:00,  1.89it/s]


14685R 46/46


  0%|          | 0/11 [00:00<?, ?it/s]/opt/miniconda3/envs/phd/lib/python3.12/site-packages/trimesh/triangles.py:302: RuntimeWarning: invalid value encountered in divide
  center_mass = integrated[1:4] / volume
100%|██████████| 11/11 [00:05<00:00,  1.91it/s]


In [91]:
df = pd.read_csv('Interference-SmoothRemesh/tpm-mc1.csv', index_col=0)
df

,50037-L,50090-R,15294-R,50053-R,50049-R,15737-R,50061-R,14726-R,50016-L,14613-R,...,50000-R,50007-L,50014-R,14827-L,14818-R,14548-R,15882-R,15282-R,50045-R,14685-R
adduction,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
abduction,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,False
flexion,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
extension,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
pinch,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
grasp,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
jar,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
pinch_load,False,False,False,False,False,True,True,False,False,False,...,False,False,False,False,False,True,True,False,False,True
grasp_load,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
jar_load,False,False,False,False,False,False,False,True,False,True,...,False,False,False,False,True,False,True,True,False,False


In [112]:
SmoothRemesh = np.array(
    [x.split('-')[0]+x.split('-')[1] for x in df.columns[~df[np.isin(df.index, poses)].any()].sort_values().to_list()]
    )

print(SmoothRemesh)

['14548R' '14613R' '14685R' '14726R' '14727R' '14818R' '14819R' '14827L'
 '14873R' '14874R' '15006R' '15283R' '15294R' '15441R' '15737R' '22306R'
 '50000R' '50001R' '50006R' '50007L' '50008L' '50014R' '50016L' '50017L'
 '50018L' '50019R' '50020R' '50021R' '50024R' '50027L' '50029R' '50034R'
 '50037L' '50045R' '50049R' '50053R']


In [113]:
from phd_helpers.paths import bone_pair_pose_intersection

poses = ['adduction','abduction','flexion','extension','pinch','grasp',
                            'jar','neutral']

smooth = bone_pair_pose_intersection(pairs, poses, 'Interference-smooth')
original = bone_pair_pose_intersection(pairs, poses, get_project_root() / '../KineticsData/FEA/CheckInterference2')

print('\noriginal')
print(original.sort_values().to_numpy())
print('\nsmooth')
print(smooth.sort_values().to_numpy())


original
['14548R' '14613R' '14685R' '14726R' '14818R' '14827L' '14873R' '15283R'
 '15441R' '22306R' '50000R' '50001R' '50006R' '50007L' '50008L' '50014R'
 '50016L' '50017L' '50018L' '50019R' '50020R' '50021R' '50024R' '50027L'
 '50029R' '50034R' '50037L' '50045R' '50049R' '50053R']

smooth
['14548R' '14613R' '14685R' '14726R' '14727R' '14818R' '14819R' '14827L'
 '14873R' '14874R' '15006R' '15283R' '15294R' '15441R' '15737R' '22306R'
 '50000R' '50001R' '50006R' '50007L' '50008L' '50014R' '50016L' '50017L'
 '50018L' '50019R' '50020R' '50021R' '50024R' '50027L' '50029R' '50034R'
 '50037L' '50045R' '50049R' '50053R']


In [123]:
print('smooth and SmoothRemesh are identical:', np.isin(smooth, SmoothRemesh).all()&(len(smooth)==len(SmoothRemesh)))

print('\nmissing from testing3 - (gave interference error in cartilage step)')
print(list(smooth.values[~np.isin(smooth.values, testing3)]))

smooth and SmoothRemesh are identical: True

missing from testing3 - (gave interference error in cartilage step)
['14613R']


All subjects in smooth / smoothRemesh should be able to get cartilage on both tpm and mc1
 - probs floors in generation code that is causing problems